# Save everything from the runs so far (no retraining)

Run this in a **separate Colab tab** at any time, even while a training run is going: it saves what is finished so far, and running it again later adds the rest.

* Trained weights and caches already live on Drive; this lists them with sizes and SHA-256.
* Small results (reports, summaries, predictions, plots) are committed to `saved_results/` on your GitHub fork.
* Optional: weights are also uploaded to a **private** Hugging Face repository as an off-Drive backup.

**Secrets (key icon, left sidebar), one time:** `GITHUB_TOKEN` — GitHub fine-grained token with *Contents: Read and write* on your fork; `HF_TOKEN` (optional) — Hugging Face token with *write* access.

In [ ]:
import os
from google.colab import drive, userdata
drive.mount('/content/drive')
REPO_DIR = '/content/zuco-multimodal-sentiment'
if not os.path.exists(REPO_DIR):
    !git clone --branch feature/neurolm-eeg-probe https://github.com/MohammadJRanjbar/zuco-multimodal-sentiment.git $REPO_DIR
%cd $REPO_DIR
!git pull --ff-only
!pip install -q huggingface_hub
RESULTS = '/content/drive/MyDrive/Thesis/Results/zuco_multimodal_sentiment'

## 1. What is already saved on Drive

In [ ]:
!find "$RESULTS" \( -name "*_weights.pt" -o -name "*.npz" \) -exec ls -lh {} \; | awk '{print $5, $9}'
!find "$RESULTS" -name "*.md" | sort

## 2. Push small results to GitHub (and optionally back up weights to Hugging Face)

Set `HF_REPO = None` to skip the Hugging Face upload.

In [ ]:
os.environ['GITHUB_TOKEN'] = userdata.get('GITHUB_TOKEN')
HF_REPO = 'MohammadJRanjbar/zuco-eeg-weights'   # private model repo; None to skip
HF_FLAG = ''
if HF_REPO:
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    HF_FLAG = f'--hf-repo {HF_REPO}'
GIT_NAME, GIT_EMAIL = 'mohammadranjbar', 'mjrkalhar@gmail.com'
!python scripts/save_results_to_github.py --name all_runs --source "$RESULTS" --push $HF_FLAG \
    --author-name "$GIT_NAME" --author-email "$GIT_EMAIL"
!cat saved_results/all_runs/LARGE_FILES.md | head -40